# Visual Classifier — Two-Stage Fine-Tuning (Local)

This notebook performs **sequential two-stage fine-tuning** of the
`dima806/ai_vs_human_generated_image_detection` ViT model **locally**
on a MacBook Air M4 using Apple Silicon MPS acceleration.

| Stage | Dataset | Purpose |
|-------|---------|--------|
| **1** | `nebula/GenImage-arrow` (small streamed subset) | Broaden the model on diverse AI generators |
| **2** | `julienlucas/midjourney-dalle-sd-nanobananapro-dataset` | Specialise on Midjourney / DALL·E / SD |

The task is **binary classification**: `0 = Real`, `1 = AI-Generated`.

### Running Locally on Apple Silicon
1. Open this notebook in VS Code (or Jupyter).
2. Select your local Python kernel with the required packages installed.
3. Training uses the MPS backend — no CUDA required.
4. All outputs are saved inside `outputs/` relative to this notebook.

> ⚠️ **Memory:** The M4 has unified memory (16–32 GB). Batch sizes are
> kept small to avoid OOM. Adjust `BATCH_SIZE` if you have more RAM.

> ⚠️ **Large model checkpoints are .gitignored.** Only lightweight deltas or adapters should be committed.

---
## 0 · Configuration
Edit the values below to control dataset sizes, training hyper-parameters, etc.

In [1]:
# ── Dataset sizes for GenImage streaming subset ──
TRAIN_PER_CLASS = 1000    # 1 000 real + 1 000 AI for training
VAL_PER_CLASS   = 250     # 250 + 250 for validation
TEST_PER_CLASS  = 500     # 500 + 500 for test
SEED            = 42

# ── Model ──
BASE_MODEL = "dima806/ai_vs_human_generated_image_detection"

# ── Training hyper-parameters ──
EPOCHS_STAGE1   = 3
EPOCHS_STAGE2   = 3
BATCH_SIZE      = 6      # M4-friendly; increase to 8 if you have 32 GB RAM
LEARNING_RATE   = 2e-5
REPLAY_RATIO    = 0.25    # fraction of Stage-1 data mixed into Stage 2

# ── Output paths ──
STAGE1_MODEL_DIR = "outputs/models/genimage_stage1_finetuned"
STAGE2_MODEL_DIR = "outputs/models/genimage_then_julienlucas_stage2_finetuned"
EVAL_OUTPUT_DIR  = "outputs"

---
## 1 · Environment Setup (Local – Apple Silicon)

In [2]:
import os, sys, torch

# ── GPU / Accelerator check ──
if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    print("✅ MPS (Apple Silicon) available")
    USE_FP16 = False   # fp16 not fully supported on MPS
elif torch.cuda.is_available():
    print(f"✅ CUDA available – {torch.cuda.get_device_name(0)}")
    USE_FP16 = True
else:
    print("⚠️  No GPU – training will be slow on CPU")
    USE_FP16 = False

print(f"Mixed precision (fp16): {USE_FP16}")
print(f"PyTorch version: {torch.__version__}")

✅ MPS (Apple Silicon) available
Mixed precision (fp16): False
PyTorch version: 2.11.0


---
## 2 · Authentication

GenImage-arrow may require a Hugging Face token.

**Do NOT hardcode your token.** Use one of these safe methods:
- Set the `HF_TOKEN` environment variable before launching the notebook.
- Or run `huggingface_hub.login()` / `notebook_login()` interactively.

In [3]:
HF_TOKEN = os.environ.get("HF_TOKEN", None)

if HF_TOKEN is None:
    try:
        from huggingface_hub import notebook_login
        notebook_login()          # interactive widget / prompt
        HF_TOKEN = True           # login() stores the token globally
    except Exception:
        print("⚠️  No HF token found. Public datasets will still work,"
              " but gated datasets may fail.")
        HF_TOKEN = None
else:
    print("✅ HF_TOKEN loaded from environment.")

---
## 3 · Imports & Suppress Warnings

In [4]:
import warnings
warnings.filterwarnings("ignore")

from transformers import AutoImageProcessor, AutoModelForImageClassification

# Our helper module (lives next to this notebook)
from src.visual_module.two_stage_finetuning import (
    get_device,
    build_genimage_subset,
    load_julienlucas_dataset,
    run_training_stage,
    evaluate_model,
)
# Weight-delta helpers from the existing visual_classifier module
from src.visual_module.visual_classifier import save_weight_delta

DEVICE = get_device()
print(f"Device: {DEVICE}")

Device: mps


---
## 4 · Load the Base Model & Processor

In [5]:
print(f"Loading base model: {BASE_MODEL}")
processor = AutoImageProcessor.from_pretrained(BASE_MODEL)
model     = AutoModelForImageClassification.from_pretrained(
    BASE_MODEL, ignore_mismatched_sizes=True
).to(DEVICE)
print(f"✅ Model loaded  –  {sum(p.numel() for p in model.parameters()):,} parameters")

Loading base model: dima806/ai_vs_human_generated_image_detection


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

✅ Model loaded  –  85,800,194 parameters


---
## 5 · Stage 1 – GenImage Subset

### 5a · Build a balanced subset via streaming

GenImage-arrow is **very large**. We stream it and collect only the
samples we need, so nothing is downloaded in full.

In [6]:
genimage_ds = build_genimage_subset(
    train_per_class=TRAIN_PER_CLASS,
    val_per_class=VAL_PER_CLASS,
    test_per_class=TEST_PER_CLASS,
    seed=SEED,
    hf_token=HF_TOKEN if isinstance(HF_TOKEN, str) else None,
)
genimage_ds

📡  Streaming GenImage-arrow (this may take a few minutes)…


Resolving data files:   0%|          | 0/1216 [00:00<?, ?it/s]

   Columns: ['image_path', 'md5', 'width', 'height', 'image']
   Example image_path: ADM/train/nature/n02966687_11178.JPEG


Resolving data files:   0%|          | 0/1216 [00:00<?, ?it/s]

   collected 500/3500…
   collected 1000/3500…
   collected 1500/3500…
   collected 2000/3500…
   collected 2500/3500…
   collected 3000/3500…
   collected 3500/3500…
   train: 2000 samples (real=1000, ai=1000)
   validation: 500 samples (real=250, ai=250)
   test: 1000 samples (real=500, ai=500)


DatasetDict({
    train: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 2000
    })
    validation: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 500
    })
    test: Dataset({
        features: ['image_path', 'md5', 'width', 'height', 'image', 'label'],
        num_rows: 1000
    })
})

### 5b · Inspect a few samples
Verify the label mapping is correct.

In [7]:
# Show 5 examples so we can confirm the label extraction logic
for i in range(min(5, len(genimage_ds["train"]))):
    ex = genimage_ds["train"][i]
    tag = "REAL" if ex["label"] == 0 else "AI"
    print(f"  [{tag}] label={ex['label']}  path={ex.get('image_path','N/A')[:80]}")

  [AI] label=1  path=Glide/train/ai/GLIDE_1000_200_08_883_glide_00141.png
  [REAL] label=0  path=stable_diffusion_v_1_4/train/nature/n02093647_7075.JPEG
  [AI] label=1  path=VQDM/train/ai/VQDM_1000_200_00_061_vqdm_00160.png
  [REAL] label=0  path=Midjourney/train/nature/n03379051_10304.JPEG
  [AI] label=1  path=Glide/train/ai/GLIDE_1000_200_07_778_glide_00140.png


In [12]:
from src.visual_module.visual_classifier import VisualClassifier

model = VisualClassifier(
    model_name_or_path="dima806/ai_vs_human_generated_image_detection",
    delta_path="src/visual_module/fine_tuned_model_delta/weight_delta.pt",
)

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Applying weight delta from 'src/visual_module/fine_tuned_model_delta/weight_delta.pt'...
Delta applied successfully.


In [14]:
metrics = evaluate_model(
    model=model.model,          # the underlying HF model
    processor=model.processor,  # the underlying processor
    test_ds=genimage_ds["test"],
    output_prefix="genimage_after_stage2",
    output_dir=EVAL_OUTPUT_DIR,
    batch_size=BATCH_SIZE,
    fp16=USE_FP16,
)

  📄  Metrics saved → outputs/genimage_after_stage2_eval_results.json
  🖼️   Confusion matrix saved → outputs/genimage_after_stage2_confusion_matrix.png

  ────────────────────────────────────────
  Accuracy  : 0.8130
  Precision : 0.8004
  Recall    : 0.8340
  F1-score  : 0.8168
  ────────────────────────────────────────

              precision    recall  f1-score   support

        Real       0.83      0.79      0.81       500
AI-Generated       0.80      0.83      0.82       500

    accuracy                           0.81      1000
   macro avg       0.81      0.81      0.81      1000
weighted avg       0.81      0.81      0.81      1000



In [15]:
from src.visual_module.visual_classifier import VisualClassifier
from PIL import Image
import os

# Load base model WITHOUT fine-tuning delta
base_model = VisualClassifier(
    model_name_or_path="dima806/ai_vs_human_generated_image_detection",
    # no delta_path
)

# Load fine-tuned model WITH delta
finetuned_model = VisualClassifier(
    model_name_or_path="dima806/ai_vs_human_generated_image_detection",
    delta_path="src/visual_module/fine_tuned_model_delta/weight_delta.pt",
)

# Test both on your sample images
sample_dir = "data/sample_images"
for fname in sorted(os.listdir(sample_dir)):
    if fname.startswith("."):
        continue
    img = Image.open(os.path.join(sample_dir, fname))
    base_result = base_model.predict(img)
    ft_result = finetuned_model.predict(img)
    
    expected = "Real" if fname.startswith("real") else "AI Generated"
    print(f"\n{fname}")
    print(f"  Expected:    {expected}")
    print(f"  Base model:  {base_result['prediction']} ({base_result['confidence']:.2%})")
    print(f"  Fine-tuned:  {ft_result['prediction']} ({ft_result['confidence']:.2%})")


Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Applying weight delta from 'src/visual_module/fine_tuned_model_delta/weight_delta.pt'...
Delta applied successfully.

fake+metadata+place-face.png
  Expected:    AI Generated
  Base model:  Real (99.06%)
  Fine-tuned:  AI Generated (53.24%)

fake+metadata-place+face.png
  Expected:    AI Generated
  Base model:  Real (99.20%)
  Fine-tuned:  AI Generated (96.98%)

fake+metadata-place-face.png
  Expected:    AI Generated
  Base model:  Real (99.01%)
  Fine-tuned:  AI Generated (54.00%)

fake-metadata+place-face.png
  Expected:    AI Generated
  Base model:  Real (99.06%)
  Fine-tuned:  AI Generated (53.24%)

fake-metadata-place+face.png
  Expected:    AI Generated
  Base model:  Real (99.20%)
  Fine-tuned:  AI Generated (96.98%)

fake-metadata-place-face.png
  Expected:    AI Generated
  Base model:  Real (99.01%)
  Fine-tuned:  AI Generated (54.00%)

real+metadata.jpeg
  Expected:    Real
  Base model:  Real (99.03%)
  Fine-tuned:  AI Generated (91.05%)

real-metadata.jpeg
  Expected:  